In [1]:
import os
import re
import json
import base64
from io import StringIO
from typing import List
from collections import Counter

from unstructured.staging.base import elements_to_json, elements_from_json
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from dotenv import load_dotenv

import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output

import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import camelot
import pikepdf
import chromadb

load_dotenv()
plt.rcParams['font.family'] = 'DejaVu Sans'

os.makedirs("extracted_data/images", exist_ok=True)
os.makedirs("extracted_data/tables", exist_ok=True)
os.makedirs("document_store", exist_ok=True)

In [2]:
def remove_pdf_restrictions(input_path, output_path=None):
    """Strip extraction-restriction flags from a PDF (some PDFs disable text extraction)"""
    if output_path is None:
        output_path = input_path.replace(".pdf", "_unlocked.pdf")
    with pikepdf.open(input_path, allow_overwriting_input=True) as pdf:
        pdf.save(output_path)
    print(f"✅ Saved unrestricted copy to {output_path}")
    return output_path

In [21]:
def partition_document(file_path: str, store_dir: str = "document_store"):
    """Extract elements from PDF, storing the parsed JSON on disk"""
    os.makedirs(store_dir, exist_ok=True)

    pdf_name = os.path.splitext(os.path.basename(file_path))[0]
    store_path = os.path.join(store_dir, f"{pdf_name}_elements.json")

    if os.path.exists(store_path):
        print(f"✅ Found stored elements at {store_path} — loading from disk")
        elements = elements_from_json(store_path)
        print(f"✅ Loaded {len(elements)} elements from disk")
        return elements

    print(f"📄 Partitioning document: {file_path} (this may take a while for hi_res)")
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )
    print(f"✅ Extracted {len(elements)} elements")

    elements_to_json(elements, filename=store_path)
    print(f"💾 Stored elements to {store_path}")
    return elements


file_path = "docs/somatosensory.pdf"
elements = partition_document(file_path)


📄 Partitioning document: docs/somatosensory.pdf (this may take a while for hi_res)


No languages specified, defaulting to English.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

✅ Extracted 59 elements
💾 Stored elements to document_store\somatosensory_elements.json


In [22]:
def find_tables_camelot(pdf_path, min_accuracy=70, min_columns=2):
    """Run Camelot as a second opinion on table detection"""
    all_tables = {}
    for flavor in ["stream", "lattice"]:
        try:
            results = camelot.read_pdf(pdf_path, pages="all", flavor=flavor)
            for t in results:
                acc = t.parsing_report.get("accuracy", 0)
                if acc >= min_accuracy and t.df.shape[1] >= min_columns:
                    all_tables.setdefault(t.page, []).append({
                        "html": t.df.to_html(index=False, header=False),
                        "accuracy": acc
                    })
        except Exception as e:
            print(f"⚠️ Camelot {flavor} failed: {e}")
    return all_tables


def score_table_confidence(html, page_text_near_table=""):
    """Heuristically score whether a Camelot-detected 'table' is real or a false positive"""
    soup = BeautifulSoup(html, 'html.parser')
    cells = [c.get_text(strip=True) for c in soup.find_all(['td', 'th'])]
    cells = [c for c in cells if c]
    if not cells:
        return 0, ["no cell content"]

    score = 0
    reasons = []

    if re.search(r'\btable\s+\d+\b', page_text_near_table, re.IGNORECASE):
        score += 40
        reasons.append("+40: 'Table N' caption found nearby")

    numeric_ratio = sum(1 for c in cells if re.search(r'\d', c)) / len(cells)
    if numeric_ratio > 0.3:
        score += 25
        reasons.append(f"+25: numeric ratio {numeric_ratio:.0%}")
    elif numeric_ratio < 0.05:
        score -= 15
        reasons.append(f"-15: almost no numbers ({numeric_ratio:.0%})")

    avg_len = sum(len(c) for c in cells) / len(cells)
    if avg_len < 25:
        score += 20
        reasons.append(f"+20: short cells (avg {avg_len:.0f} chars)")
    elif avg_len > 60:
        score -= 25
        reasons.append(f"-25: long cells, likely prose (avg {avg_len:.0f} chars)")

    word_counts = Counter(cells)
    most_common_count = word_counts.most_common(1)[0][1] if word_counts else 0
    repetition_ratio = most_common_count / len(cells)
    if repetition_ratio > 0.15:
        score -= 30
        reasons.append(f"-30: high repetition ({repetition_ratio:.0%}) — likely figure/diagram text")

    return max(0, min(100, score)), reasons


def filter_camelot_tables(camelot_tables, elements, min_score=60):
    """Auto-classify each Camelot table as real or false-positive"""
    confirmed = {}
    page_text = {}
    for el in elements:
        pg = getattr(el.metadata, 'page_number', None)
        if pg:
            page_text.setdefault(pg, "")
            page_text[pg] += " " + el.text

    for page, tables_on_page in camelot_tables.items():
        for t in tables_on_page:
            score, reasons = score_table_confidence(t['html'], page_text.get(page, ""))
            if score >= min_score:
                confirmed.setdefault(page, []).append(t)

    print(f"✅ Confirmed real tables on pages: {sorted(confirmed.keys())}")
    return confirmed


camelot_tables_raw = find_tables_camelot(file_path)
confirmed_tables = filter_camelot_tables(camelot_tables_raw, elements)

# Only keep pages unstructured actually missed
unstructured_table_pages = {
    el.metadata.page_number for el in elements
    if type(el).__name__ == "Table" and hasattr(el.metadata, "page_number")
}
recovered_tables = {
    page: [t["html"] for t in tables]
    for page, tables in confirmed_tables.items()
    if page not in unstructured_table_pages
}
print(f"📋 Tables to inject (missed by unstructured): {list(recovered_tables.keys())}")

c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (46.6929, 320.55449999999996, 552.2591, 666.8203667714286)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


✅ Confirmed real tables on pages: []
📋 Tables to inject (missed by unstructured): []


In [23]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    print(f"✅ Created {len(chunks)} chunks")
    return chunks


chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 9 chunks


In [25]:
def crop_table_from_pdf(pdf_path, page_number, element, output_path, zoom=3, padding=8):
    """Crop the exact table region from the original PDF page — pixel-perfect"""
    try:
        coords = element.metadata.coordinates
        if not coords or not coords.points:
            return None

        doc = fitz.open(pdf_path)
        page = doc[page_number - 1]
        page_rect = page.rect  # actual PDF page size, in points

        # unstructured's coordinates may be in pixel space (hi_res rendering), not
        # PDF point space — scale to match the real page dimensions
        coord_system = coords.system
        system_width = getattr(coord_system, 'width', None)
        system_height = getattr(coord_system, 'height', None)

        if system_width and system_height:
            scale_x = page_rect.width / system_width
            scale_y = page_rect.height / system_height
        else:
            scale_x = scale_y = 1.0

        xs = [p[0] * scale_x for p in coords.points]
        ys = [p[1] * scale_y for p in coords.points]
        x0, x1 = min(xs) - padding, max(xs) + padding
        y0, y1 = min(ys) - padding, max(ys) + padding

        if x1 - x0 < 5 or y1 - y0 < 5:
            doc.close()
            return None

        x0, y0 = max(x0, 0), max(y0, 0)
        x1, y1 = min(x1, page_rect.width), min(y1, page_rect.height)

        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, clip=fitz.Rect(x0, y0, x1, y1))
        pix.save(output_path)
        doc.close()
        return output_path

    except Exception as e:
        print(f"     ⚠️ Could not crop table from PDF: {e}")
        return None


def clean_cell_text(cell):
    """Extract cell text, converting <sup>/<sub> tags to unicode"""
    sup_map = {'2': '²', '3': '³', '1': '¹'}
    text = ''
    for content in cell.contents:
        if getattr(content, 'name', None) == 'sup':
            raw = content.get_text()
            text += sup_map.get(raw, f'^{raw}')
        elif getattr(content, 'name', None) == 'sub':
            text += f"_{content.get_text()}"
        else:
            text += str(content) if isinstance(content, str) else content.get_text()
    return text.strip()


def html_table_to_image(html, output_path, title=None):
    """Fallback: render table from HTML if PDF cropping isn't available"""
    try:
        soup = BeautifulSoup(html, 'html.parser')
        rows = soup.find_all('tr')
        if not rows:
            return None
        data = [[clean_cell_text(c) for c in row.find_all(['td', 'th'])] for row in rows]
        header, body = data[0], data[1:]
        if not body or not header:
            return None
        col_count = len(header)
        body = [row + [''] * (col_count - len(row)) if len(row) < col_count else row[:col_count] for row in body]
        df = pd.DataFrame(body, columns=header)
    except Exception as e:
        print(f"     ⚠️ Could not parse table HTML: {e}")
        return None

    if df.empty:
        return None

    fig, ax = plt.subplots(figsize=(max(6, len(df.columns) * 1.4), max(1.5, len(df) * 0.5 + 1)))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    tbl = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#4a4a4a')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#f5f5f5')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    return output_path

In [26]:
def unique_path(path):
    """If file already exists, append _1, _2, etc. to avoid overwriting"""
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    counter = 1
    while os.path.exists(f"{base}_{counter}{ext}"):
        counter += 1
    return f"{base}_{counter}{ext}"


def separate_content_types(chunk, chunk_id, pdf_path, recovered_tables):
    """Analyze content types AND persist images/tables to disk with paths"""
    content_data = {'text': chunk.text, 'tables': [], 'images': [], 'types': ['text'], 'page': None}

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            if content_data['page'] is None and hasattr(element.metadata, 'page_number'):
                content_data['page'] = element.metadata.page_number

            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                table_idx = len(content_data['tables'])
                table_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.html")
                with open(table_path, 'w', encoding='utf-8') as f:
                    f.write(f"<html><body>{table_html}</body></html>")

                img_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.png")
                rendered = crop_table_from_pdf(pdf_path, content_data['page'], element, img_path)
                if rendered is None:
                    rendered = html_table_to_image(table_html, img_path, title=f"Table (page {content_data['page']})")

                content_data['tables'].append({"html": table_html, "path": table_path, "image_path": rendered})

            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    img_b64 = element.metadata.image_base64
                    img_idx = len(content_data['images'])
                    img_out_path = unique_path(f"extracted_data/images/chunk_{chunk_id}_img_{img_idx}.png")
                    with open(img_out_path, 'wb') as f:
                        f.write(base64.b64decode(img_b64))
                    content_data['images'].append({"base64": img_b64, "path": img_out_path})

        # Inject Camelot-recovered tables for this chunk's page (unstructured missed these)
        if content_data['page'] in recovered_tables:
            for html in recovered_tables[content_data['page']]:
                content_data['types'].append('table')
                table_idx = len(content_data['tables'])
                table_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.html")
                with open(table_path, 'w', encoding='utf-8') as f:
                    f.write(f"<html><body>{html}</body></html>")
                img_path = unique_path(f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.png")
                rendered = html_table_to_image(html, img_path, title=f"Table (page {content_data['page']}, recovered)")
                content_data['tables'].append({"html": html, "path": table_path, "image_path": rendered})
            del recovered_tables[content_data['page']]

    content_data['types'] = list(set(content_data['types']))
    return content_data


In [27]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    try:
        model_name = "qwen/qwen3.6-27b" if images else "openai/gpt-oss-120b"
        llm = ChatGroq(model_name=model_name, temperature=0)

        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}
        """

        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
            prompt_text += """
IMPORTANT: Scan the TEXT CONTENT above for any explicit table number, label, or caption
(e.g. "Table 3"). If found, state it verbatim as the FIRST LINE of your description in
the format: "This is Table X: <topic>". Always include the exact table number if present.
"""

        prompt_text += """
        YOUR TASK:
        Generate a comprehensive, searchable description covering key facts, main topics,
        questions this content could answer, visual content analysis, and alternative search terms.

        SEARCHABLE DESCRIPTION:"""

        if images:
            message_content = [{"type": "text", "text": prompt_text}]
            for image_base64 in images:
                message_content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}})
        else:
            message_content = prompt_text

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content

    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [28]:
def save_processed_chunks(documents, store_path):
    """Save processed_chunks (LangChain Documents) to disk as JSON"""
    data = [
        {"page_content": doc.page_content, "metadata": doc.metadata}
        for doc in documents
    ]
    with open(store_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved {len(documents)} processed chunks to {store_path}")


def load_processed_chunks(store_path):
    """Load processed_chunks back into LangChain Documents"""
    with open(store_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    documents = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in data]
    print(f"✅ Loaded {len(documents)} processed chunks from disk")
    return documents


def summarise_chunks(chunks, pdf_path, recovered_tables, store_dir="document_store"):
    """Process all chunks with AI Summaries, saving images/tables to disk.
    Uses a stored JSON if available, to avoid re-running Groq summarization."""

    os.makedirs(store_dir, exist_ok=True)
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    store_path = os.path.join(store_dir, f"{pdf_name}_processed_chunks.json")

    if os.path.exists(store_path):
        print(f"✅ Found stored processed chunks at {store_path} — skipping re-summarization")
        return load_processed_chunks(store_path)

    print("🧠 Processing chunks with AI Summaries...")
    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        print(f"   Processing chunk {i+1}/{total_chunks}")
        content_data = separate_content_types(chunk, chunk_id=i, pdf_path=pdf_path, recovered_tables=recovered_tables)
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        table_htmls = [t["html"] for t in content_data['tables']]
        image_b64s = [img["base64"] for img in content_data['images']]

        if table_htmls or image_b64s:
            enhanced_content = create_ai_enhanced_summary(content_data['text'], table_htmls, image_b64s)
        else:
            enhanced_content = content_data['text']

        doc = Document(
            page_content=enhanced_content,
            metadata={
                "chunk_id": i,
                "page": content_data['page'] or 0,
                "raw_text": content_data['text'][:2000],
                "table_paths": json.dumps([t["path"] for t in content_data['tables']]),
                "table_image_paths": json.dumps([t["image_path"] for t in content_data['tables'] if t.get("image_path")]),
                "image_paths": json.dumps([img["path"] for img in content_data['images']]),
                "has_table": bool(content_data['tables']),
                "has_image": bool(content_data['images']),
            }
        )
        langchain_documents.append(doc)

    print(f"✅ Processed {len(langchain_documents)} chunks")

    save_processed_chunks(langchain_documents, store_path)
    return langchain_documents


processed_chunks = summarise_chunks(chunks, pdf_path=file_path, recovered_tables=recovered_tables)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/9
     Types found: ['text']
     Tables: 0, Images: 0
   Processing chunk 2/9
     Types found: ['image', 'text']
     Tables: 0, Images: 1
   Processing chunk 3/9
     Types found: ['image', 'text']
     Tables: 0, Images: 1
   Processing chunk 4/9
     Types found: ['text']
     Tables: 0, Images: 0
   Processing chunk 5/9
     Types found: ['text', 'table']
     Tables: 1, Images: 0
   Processing chunk 6/9
     Types found: ['text']
     Tables: 0, Images: 0
   Processing chunk 7/9
     Types found: ['text']
     Tables: 0, Images: 0
   Processing chunk 8/9
     Types found: ['text']
     Tables: 0, Images: 0
   Processing chunk 9/9
     Types found: ['text']
     Tables: 0, Images: 0
✅ Processed 9 chunks
💾 Saved 9 processed chunks to document_store\somatosensory_processed_chunks.json


In [29]:
class NomicEmbeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        return super().embed_documents([f"search_document: {t}" for t in texts])
    def embed_query(self, text):
        return super().embed_query(f"search_query: {text}")


class MultiNamespaceVectorStore:
    """A single ChromaDB with per-PDF namespace isolation via separate collections.
    Exposes similarity_search() that queries across ALL namespaces."""

    def __init__(self, persist_directory="vector_store/chroma_db"):
        self.persist_directory = persist_directory
        self.embedding_model = NomicEmbeddings(
            model_name="nomic-ai/nomic-embed-text-v1.5",
            model_kwargs={"trust_remote_code": True}
        )
        self._namespaces = {}

    def add_pdf(self, documents, pdf_name):
        """Add a PDF's chunks as a separate namespace (collection). Skips if already indexed."""
        collection_name = re.sub(r'[^a-zA-Z0-9_-]', '_', pdf_name)[:63]

        existing = Chroma(
            collection_name=collection_name,
            persist_directory=self.persist_directory,
            embedding_function=self.embedding_model,
            collection_metadata={"hnsw:space": "cosine"}
        )
        if existing._collection.count() > 0:
            print(f"✅ Namespace '{pdf_name}' already indexed ({existing._collection.count()} chunks) — skipping")
            self._namespaces[pdf_name] = existing
            return

        for doc in documents:
            doc.metadata["source_pdf"] = pdf_name

        doc_ids = [f"{collection_name}_chunk_{i}" for i in range(len(documents))]

        print(f"🔮 Indexing '{pdf_name}' into namespace ({len(documents)} chunks)...")
        vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embedding_model,
            persist_directory=self.persist_directory,
            collection_name=collection_name,
            collection_metadata={"hnsw:space": "cosine"},
            ids=doc_ids
        )
        print(f"✅ Namespace '{pdf_name}' indexed")
        self._namespaces[pdf_name] = vectorstore

    def similarity_search(self, query, k=15):
        """Search across ALL namespaces, return top-k combined results."""
        all_results = []
        per_ns_k = max(k, 10)

        for name, store in self._namespaces.items():
            try:
                results = store.similarity_search_with_relevance_scores(query, k=per_ns_k)
                all_results.extend(results)
            except Exception as e:
                print(f"⚠️ Search failed in namespace '{name}': {e}")

        all_results.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, score in all_results[:k]]

    def delete_namespace(self, pdf_name):
        """Remove a PDF's entire namespace"""
        
        collection_name = re.sub(r'[^a-zA-Z0-9_-]', '_', pdf_name)[:63]
        client = chromadb.PersistentClient(path=self.persist_directory)
        try:
            client.delete_collection(collection_name)
            self._namespaces.pop(pdf_name, None)
            print(f"🗑️ Deleted namespace '{pdf_name}'")
        except Exception as e:
            print(f"⚠️ Could not delete namespace '{pdf_name}': {e}")

    def list_namespaces(self):
        """List all indexed PDFs"""
    
        client = chromadb.PersistentClient(path=self.persist_directory)
        collections = client.list_collections()
        for c in collections:
            print(f"  📄 {c.name} ({c.count()} chunks)")
        return collections


# --- Process all PDFs and build the shared vector store ---
pdf_files = [
    "docs/attention-is-all-you-need.pdf",
    # Add more PDFs here:
    "docs/somatosensory.pdf",
    # "docs/sample-tables.pdf",
]

db = MultiNamespaceVectorStore()
all_processed_chunks = []

for file_path in pdf_files:
    elements = partition_document(file_path)

    camelot_tables_raw = find_tables_camelot(file_path)
    confirmed_tables = filter_camelot_tables(camelot_tables_raw, elements)
    unstructured_table_pages = {
        el.metadata.page_number for el in elements
        if type(el).__name__ == "Table" and hasattr(el.metadata, "page_number")
    }
    recovered_tables = {
        page: [t["html"] for t in tables]
        for page, tables in confirmed_tables.items()
        if page not in unstructured_table_pages
    }

    chunks = create_chunks_by_title(elements)
    processed = summarise_chunks(chunks, pdf_path=file_path, recovered_tables=recovered_tables)

    # Tag each chunk with source_pdf
    pdf_name = os.path.splitext(os.path.basename(file_path))[0]
    for doc in processed:
        doc.metadata["source_pdf"] = pdf_name

    all_processed_chunks.extend(processed)
    db.add_pdf(processed, pdf_name)

print(f"\n📊 Total chunks across all PDFs: {len(all_processed_chunks)}")


<All keys matched successfully>


✅ Found stored elements at document_store\attention-is-all-you-need_elements.json — loading from disk
✅ Loaded 266 elements from disk


c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (98.0, 94.5072576, 514.3140640576001, 234.60110706666666)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (98.0, 659.6790784, 513.9972105879998, 768.325786877612)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


✅ Confirmed real tables on pages: [8, 9, 10]
🔨 Creating smart chunks...
✅ Created 33 chunks
✅ Found stored processed chunks at document_store\attention-is-all-you-need_processed_chunks.json — skipping re-summarization
✅ Loaded 33 processed chunks from disk
✅ Namespace 'attention-is-all-you-need' already indexed (33 chunks) — skipping
✅ Found stored elements at document_store\somatosensory_elements.json — loading from disk
✅ Loaded 59 elements from disk


c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (46.6929, 320.55449999999996, 552.2591, 666.8203667714286)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


✅ Confirmed real tables on pages: []
🔨 Creating smart chunks...
✅ Created 9 chunks
✅ Found stored processed chunks at document_store\somatosensory_processed_chunks.json — skipping re-summarization
✅ Loaded 9 processed chunks from disk
🔮 Indexing 'somatosensory' into namespace (9 chunks)...
✅ Namespace 'somatosensory' indexed

📊 Total chunks across all PDFs: 42


In [30]:
def build_bm25_index(processed_chunks):
    """Build a keyword-search index alongside the vector store"""
    tokenized = [
        (doc.page_content + " " + doc.metadata.get("raw_text", "")).lower().split()
        for doc in processed_chunks
    ]
    bm25 = BM25Okapi(tokenized)
    print(f"✅ BM25 index built over {len(processed_chunks)} chunks")
    return bm25


bm25_index = build_bm25_index(all_processed_chunks)

✅ BM25 index built over 42 chunks


In [31]:
print("🔄 Loading reranker model...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Reranker loaded")

🔄 Loading reranker model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker loaded


In [32]:
def hybrid_search(query, db, bm25_index, processed_chunks, k=15, alpha=0.5):
    """Combine semantic (Chroma) and keyword (BM25) search via Reciprocal Rank Fusion.
    alpha: weight toward semantic (1.0) vs keyword (0.0); 0.5 = balanced"""

    semantic_results = db.similarity_search(query, k=k)
    bm25_scores = bm25_index.get_scores(query.lower().split())
    bm25_ranked_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]

    rrf_scores = {}
    for rank, doc in enumerate(semantic_results):
        cid = (doc.metadata.get("source_pdf", ""), doc.metadata.get("chunk_id"))
        rrf_scores[cid] = rrf_scores.get(cid, 0) + alpha * (1 / (rank + 60))
    for rank, idx in enumerate(bm25_ranked_idx):
        cid = (processed_chunks[idx].metadata.get("source_pdf", ""), processed_chunks[idx].metadata.get("chunk_id"))
        rrf_scores[cid] = rrf_scores.get(cid, 0) + (1 - alpha) * (1 / (rank + 60))

    top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]
    id_to_doc = {(d.metadata.get("source_pdf", ""), d.metadata.get("chunk_id")): d for d in processed_chunks}
    return [id_to_doc[i] for i in top_ids if i in id_to_doc]


def rerank(query, candidates, top_k=5):
    """Cross-encoder reranking for more precise ordering of top candidates"""
    if not candidates:
        return []
    pairs = [(query, doc.page_content) for doc in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked[:top_k]]


def retrieve(query, db, bm25_index, processed_chunks, k=5):
    """Full retrieval pipeline: hybrid search -> rerank -> top k"""
    candidates = hybrid_search(query, db, bm25_index, processed_chunks, k=15)
    return rerank(query, candidates, top_k=k)


In [33]:
def generate_final_answer(chunks, query):
    """Generate final answer + structured source list"""
    sources = []
    has_images = False

    try:
        prompt_text = f"Based on the following documents, please answer this question: {query}\n\nCONTENT TO ANALYZE:\n"

        for i, chunk in enumerate(chunks):
            meta = chunk.metadata
            prompt_text += f"--- Document {i+1} (page {meta.get('page')}) ---\nTEXT:\n{meta.get('raw_text', chunk.page_content)}\n\n"

            table_paths = json.loads(meta.get("table_paths", "[]"))
            table_image_paths = json.loads(meta.get("table_image_paths", "[]"))
            image_paths = json.loads(meta.get("image_paths", "[]"))

            if table_paths:
                prompt_text += "TABLES:\n"
                for p in table_paths:
                    with open(p, 'r', encoding='utf-8') as f:
                        prompt_text += f.read() + "\n\n"

            source_entry = {"chunk_id": meta.get("chunk_id"), "page": meta.get("page"), "type": "text", "preview": chunk.page_content[:150], "paths": []}
            if table_paths:
                source_entry["type"] = "table"
                source_entry["paths"].extend(table_image_paths if table_image_paths else table_paths)
            if image_paths:
                has_images = True
                source_entry["type"] = "image" if not table_paths else "table+image"
                source_entry["paths"].extend(image_paths)

            sources.append(source_entry)
            prompt_text += "\n"

        prompt_text += '\nPlease provide a clear, comprehensive answer using the text, tables, and images above. If the documents don\'t contain sufficient information, say "I don\'t have enough information to answer that question based on the provided documents."\n\nANSWER:'

        model_name = "qwen/qwen3.6-27b" if has_images else "openai/gpt-oss-120b"
        llm = ChatGroq(model_name=model_name, temperature=0)

        if has_images:
            message_content = [{"type": "text", "text": prompt_text}]
            for chunk in chunks:
                for path in json.loads(chunk.metadata.get("image_paths", "[]")):
                    with open(path, 'rb') as f:
                        img_b64 = base64.b64encode(f.read()).decode('utf-8')
                    message_content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}})
        else:
            message_content = prompt_text

        response = llm.invoke([HumanMessage(content=message_content)])
        answer_text = response.content

        if "don't have enough information" in answer_text.lower():
            sources = []

        return answer_text, sources

    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer.", sources


In [34]:
def render_table_html(raw_html):
    return f"""
    <style>
        .rag-table-wrapper {{ font-family: -apple-system, sans-serif; font-size: 13px; overflow-x: auto; margin: 10px 0; }}
        .rag-table-wrapper table {{ border-collapse: collapse; width: 100%; }}
        .rag-table-wrapper th, .rag-table-wrapper td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
        .rag-table-wrapper th {{ background-color: #f0f0f0; font-weight: 600; }}
        .rag-table-wrapper tr:nth-child(even) {{ background-color: #fafafa; }}
    </style>
    <div class="rag-table-wrapper">{raw_html}</div>
    """


def display_query_result(query, answer, sources):
    print(f"❓ Query: {query}\n")
    print(f"💡 Answer:\n{answer}\n")
    print(f"📚 Sources ({len(sources)}):\n")

    output_area = widgets.Output()

    def make_click_handler(source):
        def handler(b):
            with output_area:
                clear_output(wait=True)
                print(f"--- chunk {source['chunk_id']} | page {source['page']} | type: {source['type']} ---\n")
                if not source['paths']:
                    print(source['preview'])
                for path in source['paths']:
                    if path.endswith(('.png', '.jpg', '.jpeg')):
                        display(IPImage(filename=path))
                    elif path.endswith('.html'):
                        with open(path, 'r', encoding='utf-8') as f:
                            display(HTML(render_table_html(f.read())))
        return handler

    buttons = [widgets.Button(description=f"[{i+1}] page {s['page']} • {s['type']}", layout=widgets.Layout(width='auto')) for i, s in enumerate(sources)]
    for btn, source in zip(buttons, sources):
        btn.on_click(make_click_handler(source))

    display(widgets.HBox(buttons))
    display(output_area)

In [35]:
query = "in spinal cord (middle) and expanded schematic (right) The spindle is a stretch receptor with its own motor supply"

retrieved_chunks = retrieve(query, db, bm25_index, all_processed_chunks, k=5)
answer, sources = generate_final_answer(retrieved_chunks, query)
display_query_result(query, answer, sources)

❓ Query: in spinal cord (middle) and expanded schematic (right) The spindle is a stretch receptor with its own motor supply

💡 Answer:

<think>
The user wants me to answer a question based on the provided text and images.
The question is: "in spinal cord (middle) and expanded schematic (right) The spindle is a stretch receptor with its own motor supply"

Looking at the provided text, specifically Document 1 (page 2), there is a caption for "Figure 2".
The caption reads: "Figure 2: Mammalian muscle spindle showing typical position in a muscle (left), neuronal connections in spinal cord (middle) and expanded schematic (right). The spindle is a stretch receptor with its own motor supply consisting of several intrafusal muscle fibres. The sensory endings of a primary (group Ia) afferent and a secondary (group II) afferent coil around the non-contractile central portions of the intrafusal fibres."

The question seems to be asking for the completion of the sentence or a description based on 

Output()